# Lesson 13 - Agent Memory


## Setup

Dis notebook dey show how to build travel booking agent with **persistent memory** using **Microsoft Agent Framework** (MAF).

You go learn how different kinds of agent memory — working, short-term, and long-term — dey affect how agent dey keep and use information during conversation.

**Prerequisites:**
- Azure AI Foundry project wey get deployed chat model (e.g. `gpt-4o-mini`).
- Logged in with Azure CLI — run `az login` for your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Kain Agent Memory Dem Get

AI agents fit use different kain memory, wey each get im own purpose:

### Working Memory
Na di conversation thread itsef — di messages wey dem dey exchange for one session. Di agent fit look back for earlier messages for di same thread to make sure say e still make sense. For MAF dis one na im **`agent.create_session()`** dey create, wey e go return one `AgentSession`.

### Short-Term Memory
Information wey go stay for di duration of one task or session but no go save for ever. For example, di agent fit gather facts during one multi-turn planning conversation and use dem to produce final itinerary.

### Long-Term Memory
Preferences and facts wey go last **across sessions**. Person wey don ever use am before no suppose dey repeat dia dietary restrictions or travel style. Long-term memory dey usually dey supported by outside store — like database, file, or vector index — and tools go show am for agent.


## Working Memory wit Sessions

Di simplest kind memory na di conversation session. Wen you pass di same session (wey you create via `agent.create_session()`) go di next `agent.run()` calls, di agent go see di whole history of dat conversation and fit remember wetin happen before.

Make we create travel agent and show how working memory dey work.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

The agent correctly recall di budget becos both messages dey share di same session. Dis na **working memory** — e just dey exist for di lifetime of di session.

### Wetin go happen if we start new thread?

If we create **new** session, di agent no go get memory of di previous conversation:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Long-Term Memory Pattern

To remember user preferences **across sessions**, we need a persistent store wey dey outside the conversation thread. The agent dey access this store through **tools** — na functions e fit call to save and find information.

Below we dey implement one simple in-memory preference store (for production you for back am with database or vector index) and expose am as tools wey the agent fit use.

### Architecture
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — First-time user book anniversary trip

Sarah come visit for the first time. The agent go store her preferences through the tools and use am to recommend hotels.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah return weeks later

Sarah start **brand-new thread** (wey mimic new session). The working memory empty, but the long-term preference store still get her information. The agent suppose comot am make use am to personalize recommendations.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

For dis lesson, you learn three kain agent memory and how to take implement dem wit di Microsoft Agent Framework:

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` | Single conversation |
| **Short-term** | Accumulated context within a thread | Single task / session |
| **Long-term** | External store accessed via `@tool` functions | Across sessions |

### Key Takeaways
1. **`agent.create_session()`** dey provide working memory — di agent go fit see di full conversation history inside one session.
2. **New sessions no get context again** — without long-term memory, di agent no go fit remember wetin dem talk before.
3. **`@tool` functions** be di bridge — dem dey allow di agent save and pick information from persistent store.
4. **Personalization dey improve as time dey pass** — di more preferences wey dem store, di beta recommendations wey di agent fit give.

### Real-World Applications
- **Customer Service**: Make e dey remember customer history and wetin dem like
- **Personal Assistants**: Make e hold context across days or weeks
- **Healthcare**: Track patient information and wetin dem prefer
- **E-commerce**: Personalized shopping based on history

### Next Steps
- Change di in-memory dict to database or vector store (like Azure AI Search)
- Add memory expiration for time wey sensitive information dey
- Build multi-agent systems wey dey share memory
- Explore di Cognee notebook for knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dis documento na AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator) translate am. Even tho we try make e correct, abeg sabi say automated translations fit get some errors or mistake. Di original documento wey e dey for im own language na di correct one. For important info, make person wey sabi do professional human translation handle am. We no go responsible if person no understand well or if mistake show from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
